# FedMed-BN: Notebook 2 - Centralized Baseline Training
**Train BanglaBERT on ALL data combined (non-federated)**

This gives you the upper-bound performance to compare federated results against.

In [ ]:
# Cell 1: Imports & Setup
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, AdamW
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import numpy as np
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# Cell 2: Load label mappings & data
with open('/content/FedMed-BN/data/tag2id.json', 'r') as f:
    tag2id = json.load(f)
with open('/content/FedMed-BN/data/id2tag.json', 'r') as f:
    id2tag = {int(k): v for k, v in json.load(f).items()}

def load_bio_file(filepath):
    sentences = []
    current = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    current.append((parts[0], parts[1]))
    if current:
        sentences.append(current)
    return sentences

train_sentences = load_bio_file('/content/FedMed-BN/data/centralized_train.txt')
test_sentences = load_bio_file('/content/FedMed-BN/data/centralized_test.txt')

print(f"Train: {len(train_sentences)} sentences")
print(f"Test: {len(test_sentences)} sentences")

In [ ]:
# Cell 3: Load BanglaBERT tokenizer & model
MODEL_NAME = "csebuetnlp/banglabert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

num_labels = len(tag2id)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2tag,
    label2id=tag2id
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of labels: {num_labels}")
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Cell 4: Tokenize & Align Labels for NER
MAX_LENGTH = 128

def tokenize_and_align_labels(sentences, tokenizer, max_length=MAX_LENGTH):
    # Convert sentences to tokenized format with aligned labels
    tokenized_inputs = {
        'input_ids': [],
        'attention_mask': [],
        'labels': []
    }
    
    for tokens, tags in sentences:
        # Tokenize
        encoding = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors=None
        )
        
        # Align labels with subwords
        word_ids = encoding.word_ids()
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(tag2id.get(tags[word_idx], tag2id['O']))
            else:
                label_ids.append(-100)  # Subword tokens
            previous_word_idx = word_idx
        
        tokenized_inputs['input_ids'].append(encoding['input_ids'])
        tokenized_inputs['attention_mask'].append(encoding['attention_mask'])
        tokenized_inputs['labels'].append(label_ids)
    
    return tokenized_inputs

train_encodings = tokenize_and_align_labels(train_sentences, tokenizer)
test_encodings = tokenize_and_align_labels(test_sentences, tokenizer)

print(f"Train samples: {len(train_encodings['input_ids'])}")
print(f"Test samples: {len(test_encodings['input_ids'])}")

In [ ]:
# Cell 5: Create PyTorch Datasets
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.encodings['labels'][idx])
        }
    
    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = NERDataset(train_encodings)
test_dataset = NERDataset(test_encodings)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Cell 6: Training Loop
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 5
best_f1 = 0

def compute_metrics(preds, labels):
    # Compute seqeval metrics
    true_predictions = [
        [id2tag[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]
    
    results = {
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions),
        'f1': f1_score(true_labels, true_predictions),
    }
    return results, true_labels, true_predictions

for epoch in range(EPOCHS):
    # Training
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    avg_train_loss = total_loss / len(train_loader)
    
    # Evaluation
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = outputs.logits.argmax(dim=-1)
            
            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    metrics, true_labels, true_preds = compute_metrics(all_preds, all_labels)
    
    print(f"\nEpoch {epoch+1}: Train Loss = {avg_train_loss:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1: {metrics['f1']:.4f}")
    
    # Detailed per-entity report
    print(classification_report(true_labels, true_preds, digits=4))
    
    # Save best model
    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        model.save_pretrained('/content/FedMed-BN/models/centralized/best_model')
        tokenizer.save_pretrained('/content/FedMed-BN/models/centralized/best_model')
        print(f"  *** New best model saved! F1 = {best_f1:.4f} ***")

In [ ]:
# Cell 7: Save final results
import json

results = {
    'centralized': {
        'best_f1': best_f1,
        'epochs': EPOCHS,
        'model': MODEL_NAME,
        'num_labels': num_labels
    }
}

with open('/content/FedMed-BN/results/centralized_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Centralized baseline training complete!")
print(f"Best F1: {best_f1:.4f}")
print("Model saved to /content/FedMed-BN/models/centralized/best_model")
print("Results saved to /content/FedMed-BN/results/centralized_results.json")